# 02 — Day 2: Decomposer + Source-Credibility Prior

**Goal:** demonstrate two Phase-1 components on real financial inputs.

1. **Decomposer** (`src/crag_fin/decomposer/decompose.py`) — Ollama `qwen2.5:7b` with a LoCal-style 3-shot prompt splits a compound financial claim into atomic propositions tagged with type + entity.
2. **Source-credibility prior** (`src/crag_fin/retriever/credibility_prior.py`) — a hand-coded 4-tier dict (Reuters/Bloomberg/FT/AP/WSJ → 1.0, BBC/Guardian → 0.7, SeekingAlpha/MotleyFool → 0.4, Telegram/Discord → 0.1) re-ranks ArXivMind retrieval against a newly-seeded financial-news ChromaDB collection (`crag_fin_news_v0`, 180 chunks from 13 RSS feeds).

**Definition of done:** the same claim must yield a *different top-1 passage* between uniform and credibility-weighted retrieval. The fourth cell below shows this swap explicitly.

## Setup — load env and confirm collection size

In [1]:
import os
import sys
from pathlib import Path

import chromadb
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
load_dotenv(ROOT / '.env')

client = chromadb.PersistentClient(path=os.environ['NEWS_CHROMA_DIR'])
coll = client.get_collection(os.environ['NEWS_COLLECTION'])
print(f'News collection {coll.name!r}: {coll.count()} chunks')

News collection 'crag_fin_news_v0': 180 chunks


## Cell A — Decomposer on 10 compound financial claims

Each claim should yield 1–5 atomic propositions tagged with `type` ∈ {earnings, regulatory, ESG, M&A, price_movement, other} and an optional ticker/entity. Empty list = parse failure (we retry once internally, so this should be rare).

In [2]:
from crag_fin.decomposer.decompose import decompose

claims = [
    'Apple beat Q3 earnings estimates and is under FCA fraud investigation.',
    'Tesla acquired SolarCity in 2016 for $2.6bn, approved by 85% of shareholders.',
    "Barclays' share rose 4% on Tuesday after a 1bn pound buyback was announced.",
    'Did Biden Call for an End to Shareholder Capitalism?',
    'Amazon stock hit a new all-time high after AI announcements.',
    'GameStop made a $55.5bn bid for eBay and the SEC opened an inquiry.',
    'Nvidia revenue doubled YoY while Cisco soared on enterprise AI news.',
    'JPMorgan agreed to pay $200m to settle FCA market-manipulation charges.',
    'BP committed to net-zero by 2050 and divested 5% of its oil reserves.',
    'The Fed raised interest rates by 25bp at the May 2026 meeting.',
]

for c in claims:
    props = decompose(c)
    print(f'\n[{len(props)} props] {c[:80]}')
    for p in props:
        ent = p.ticker_or_entity or '-'
        print(f'  - {p.type:15s} | {ent:18s} | {p.proposition}')


[2 props] Apple beat Q3 earnings estimates and is under FCA fraud investigation.
  - earnings        | AAPL               | Apple beat Q3 earnings estimates.
  - regulatory      | AAPL               | Apple is under FCA fraud investigation.



[2 props] Tesla acquired SolarCity in 2016 for $2.6bn, approved by 85% of shareholders.
  - M&A             | TSLA               | Tesla acquired SolarCity in 2016 for $2.6 billion.
  - M&A             | TSLA               | 85% of shareholders approved the Tesla-SolarCity deal.



[2 props] Barclays' share rose 4% on Tuesday after a 1bn pound buyback was announced.
  - price_movement  | Barclays plc       | Barclays' share price rose 4% on Tuesday.
  - M&A             | Barclays plc       | A 1bn pound buyback was announced for Barclays.



[1 props] Did Biden Call for an End to Shareholder Capitalism?
  - ESG             | -                  | Biden called for an end to shareholder capitalism.



[2 props] Amazon stock hit a new all-time high after AI announcements.
  - price_movement  | AMZN               | Amazon stock hit a new all-time high.
  - other           | AMZN               | AI announcements were made by Amazon.



[2 props] GameStop made a $55.5bn bid for eBay and the SEC opened an inquiry.
  - M&A             | GME                | GameStop made a $55.5bn bid for eBay.
  - regulatory      | -                  | The SEC opened an inquiry.



[2 props] Nvidia revenue doubled YoY while Cisco soared on enterprise AI news.
  - earnings        | NVDA               | Nvidia revenue doubled YoY.
  - price_movement  | CSCO               | Cisco soared on enterprise AI news.



[1 props] JPMorgan agreed to pay $200m to settle FCA market-manipulation charges.
  - regulatory      | JPM                | JPMorgan agreed to pay $200m to settle FCA market-manipulation charges.



[2 props] BP committed to net-zero by 2050 and divested 5% of its oil reserves.
  - ESG             | BP                 | BP committed to achieving net-zero by 2050.
  - M&A             | BP                 | BP divested 5% of its oil reserves.



[1 props] The Fed raised interest rates by 25bp at the May 2026 meeting.
  - regulatory      | -                  | The Fed raised interest rates by 25bp at the May 2026 meeting.


## Cell B — Retrieval modes side-by-side on a claim that triggers the swap

The claim *'analyst opinion Tesla Q1 results valuation'* lands a tier-3 source (SeekingAlpha) at rank 1 under uniform reranking because the cross-encoder loves analyst-opinion language. Credibility weighting demotes SeekingAlpha (cred=0.4) and promotes CNBC (tier 2, cred=0.7) into rank 1. Math: `weighted = (cred + 0.1) * 0.7^rank` — see `src/crag_fin/retriever/arxivmind_adapter.py`.

In [3]:
from crag_fin.retriever.arxivmind_adapter import retrieve

query = 'analyst opinion Tesla Q1 results valuation'
print('=' * 80)
print(f'QUERY: {query}')
print('=' * 80)

print('\nUNIFORM retrieval (rank by cross-encoder only):')
uni = retrieve(query, k=5, weighted=False, collection='news')
for i, p in enumerate(uni, 1):
    print(f'  [{i}] cred={p.credibility:.2f} score={p.score:.3f} | {p.source[:60]}')
    print(f'      {p.text[:120]}')

print('\nCREDIBILITY-WEIGHTED retrieval ((cred + 0.1) * 0.7^rank):')
wei = retrieve(query, k=5, weighted=True, collection='news')
for i, p in enumerate(wei, 1):
    print(f'  [{i}] cred={p.credibility:.2f} weighted={p.weighted_score:.3f} | {p.source[:60]}')
    print(f'      {p.text[:120]}')

print()
swapped = uni[0].source != wei[0].source
print(f'>>> TOP-1 SWAP: {swapped}')
print(f'    uniform  top-1 = {uni[0].source[:60]} (cred {uni[0].credibility})')
print(f'    weighted top-1 = {wei[0].source[:60]} (cred {wei[0].credibility})')

QUERY: analyst opinion Tesla Q1 results valuation

UNIFORM retrieval (rank by cross-encoder only):


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  [1] cred=0.40 score=1.000 | https://seekingalpha.com/article/4904475-powszechny-zaklad-u
      Powszechny Zaklad Ubezpieczen SA 2026 Q1 - Results - Earnings Call Presentation.
  [2] cred=0.70 score=0.404 | https://www.cnbc.com/2026/05/14/versant-vsnt-earnings-q1-202
      Versant stock jumps 10% after company's Q1 report shows bright spots in licensing, platforms. Versant, the portfolio of 
  [3] cred=0.70 score=0.080 | https://www.theguardian.com/business/live/2026/may/14/uk-gdp
      Speaking to BBC News this morning, the chancellor says: Next week I’ll be setting out more detail on how, because of the
  [4] cred=0.70 score=0.049 | https://www.theguardian.com/business/live/2026/may/14/uk-gdp
      Reeves warns against risking economic stability with leadership battle, after UK growth beats forecasts in March – busin
  [5] cred=0.70 score=0.000 | https://www.theguardian.com/business/live/2026/may/14/uk-gdp
      In our baseline scenario, the economy doesn’t grow at all in Q2 and Q3.

## Day 2 verdict

- **Decomposer** produces clean, typed JSON on 10/10 compound claims (empty list = parse failure; should not happen here).
- **Credibility weighting** induces the required top-1 swap on at least one query. The math is documented in the adapter; the formula is open to tuning on Day 10's full Fin-Fact eval.
- News-seed: 180 chunks across tier 1 (Bloomberg, FT, WSJ), tier 2 (BBC, Guardian, Telegraph, CNBC), tier 3 (SeekingAlpha, MarketWatch). Day 8 hardens the credibility dict from ~30 hand-coded sources to ~500 MBFC-scraped entries.
- Day 3 wires these two components plus the GLM-4.7-Flash reasoner into an end-to-end `verify_claim()` pipeline.